Assignment: The Optimal Sizing Problem
Quantitative Portfolio Construction & Transaction Costs
Objective:

Install necessary libraries: yfinance, plotly, scipy.

Fetch real market data for a basket of assets.

Implement the Optimal Sizing Equation from the presentation's final slide.

Visualize how transaction costs and risk aversion change the "Target" portfolio weights.

In [6]:
# imports
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from scipy.optimize import minimize

# Parameters (Consistent with day_1.ipynb style)
TICKERS = ["AAPL", "TSLA", "BTC-USD", "GLD"]
START_DATE = "2023-01-01"
END_DATE = "2024-01-01"

# Optimization Parameters (From Slide 23)
LAMBDA = 2.5   # Risk Aversion
KAPPA = 1.0    # Cost Sensitivity

In [8]:
# Download data
print(f"Downloading data for: {', '.join(TICKERS)}")
data = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust = True)["Close"]

# Calculate Daily Returns
returns = data.pct_change().dropna()

# Expected Returns (Annualized mean) and Covariance (Annualized)
mu = returns.mean() * 252
sigma = returns.cov() * 252

print("\nAnnualized Expected Returns:")
print(mu)

[                       0%                       ]

[*********************100%***********************]  4 of 4 completed


Annualized Expected Returns:
Ticker
AAPL       0.318061
BTC-USD    0.713608
GLD        0.083524
TSLA       0.674753
dtype: float64



C:\Users\marco\AppData\Local\Temp\ipykernel_39756\2226092685.py:6: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = data.pct_change().dropna()


In [9]:
# Current portfolio (assume equal weights for now)
w_prev = np.array([1.0 / len(TICKERS)] * len(TICKERS))

def objective(w, mu, sigma, w_prev, lam, kappa):
    # 1. Alpha Term (Expected Return)
    alpha = np.dot(w, mu)
    
    # 2. Risk Penalty (Variance adjusted by Lambda)
    risk = (lam / 2) * np.dot(w.T, np.dot(sigma, w))
    
    # 3. Transaction Cost Term (Absolute difference scaled by Kappa)
    # Using a 10bps linear cost estimate
    costs = kappa * np.sum(np.abs(w - w_prev) * 0.001)
    
    # We minimize the negative to maximize the objective
    return -(alpha - risk - costs)

# Constraints: Weights sum to 1.0 (Fully Invested)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})
bounds = tuple((0, 1) for _ in range(len(TICKERS)))

# Solve
result = minimize(objective, w_prev, args=(mu, sigma, w_prev, LAMBDA, KAPPA),
                  method='SLSQP', bounds=bounds, constraints=constraints)

w_optimal = result.x

In [11]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=TICKERS,
    y=w_prev,
    name='Initial Weights (Equal)',
    marker_color='indianred'
))

fig.add_trace(go.Bar(
    x=TICKERS,
    y=w_optimal,
    name='Optimal Weights (Net of Costs)',
    marker_color='lightsalmon'
))

fig.update_layout(
    title="Portfolio Rebalancing: Initial vs. Optimal Sizing",
    xaxis_title="Assets",
    yaxis_title="Weight (%)",
    template="plotly_dark",
    barmode='group'
)

fig.show()